In [ ]:
import os, sys
import pandas as pd
import torch
import numpy as np
file_path = "./data/MPRA_eQTL.vcf"
df = pd.read_csv("./data/MPRA_eQTL.tsv", sep='\t', header=0)
print(df.head())

In [ ]:
# If using a MAC
if "qnnpack" in torch.backends.quantized.supported_engines:
    torch.backends.quantized.engine = "qnnpack"

In [ ]:
import seilora as sl
rank = 256
model = sl.Sei_LLRA(k=rank, projection=True, mode = "variant", device="cpu")

In [ ]:
from tqdm import tqdm
from sei_lora.dataloaders import VariantDataset, VariantDataLoader
dataset = VariantDataset(file_path=file_path, fasta_path = "./resources/hg38_UCSC.fa", window_size = 4096)
dataloader = VariantDataLoader(dataset=dataset, batch_size=32, shuffle=False, num_workers=0)
model.eval()

all_cp_ref = []
all_cp_alt = []
all_vcf = []

progress_bar = tqdm(dataloader, desc=f"Running {rank} benchmark")

for batch in progress_bar:
    ref, alt, vcf = batch


    cp_outputs = model((ref, alt))  # both are tuples: (refproj, altproj)


    all_cp_ref.append(cp_outputs[0])
    all_cp_alt.append(cp_outputs[1])
    all_vcf.append(vcf)

    # Accumulate by appending to list

all_cp_ref = torch.cat([t.detach() for t in all_cp_ref], dim=0).numpy()
all_cp_alt = torch.cat([t.detach() for t in all_cp_alt], dim=0).numpy()

all_vcf = np.concatenate(all_vcf, axis=0)



In [ ]:
from sklearn.metrics import roc_auc_score
sc_diff = all_cp_alt - all_cp_ref

df_pred = pd.DataFrame(all_vcf, columns=["CHROM", "POS", "NAME", "REF", "ALT"])
df_pred["POS"] = df_pred["POS"].astype(int)

seqclass_path = os.path.join( "./resources/seqclass.names")
with open(seqclass_path, "r") as f:
    sc_names = []
    for line in f:
        parts = line.strip().split()
        if len(parts) > 1:
            sc_names.append("-".join(parts[1:]))
        else:
            sc_names.append(parts[0])

df_sc = pd.DataFrame(sc_diff[:, :40], columns=sc_names[:40])

df_pred =  pd.concat([df_pred, df_sc], axis=1)


df_ou = df[df['consequence'].isin(['over', 'under'])].copy()
df_combine_ou = df_ou.merge(df_pred, left_on = ["chrom", "pos", "ref", "alt"], right_on=["CHROM", "POS", "REF", "ALT"], how = "inner")
binary_labels_ou = (df_combine_ou['consequence'] == 'over')
roc_promoter_ou = roc_auc_score(binary_labels_ou, df_combine_ou["Promoter"])
print(roc_promoter_ou)
